In [9]:
import math
import numpy as np
import scipy.stats as st
import scipy.optimize as opt

np.random.seed(52)

eps = 0.05
n = 100
k = 10
freq = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7], dtype=float)
bounds = np.arange(1, 10)


# критерий хи-квадрат
def grouped_probs(a: float, sigma: float) -> np.ndarray | None:
    if sigma <= 0:
        return None

    cdf_vals = st.norm.cdf(bounds, loc=a, scale=sigma)

    probs = np.empty(10)
    probs[0] = cdf_vals[0]
    for i in range(1, 9):
        probs[i] = cdf_vals[i] - cdf_vals[i - 1]
    probs[9] = 1.0 - cdf_vals[-1]

    if np.any(probs <= 0) or not np.isclose(probs.sum(), 1.0, atol=1e-8):
        return None

    return probs


def neg_loglik(params: np.ndarray) -> float:
    a, sigma = params
    probs = grouped_probs(a, sigma)

    if probs is None:
        return np.inf

    return -np.sum(freq * np.log(probs))


fit = opt.differential_evolution(
    func=neg_loglik,
    bounds=[(0.0, 10.0), (0.1, 10.0)],
    maxiter=10000,
    seed=52
)

a_mle = float(fit.x[0])
sigma_mle = float(fit.x[1])

probs_hat = grouped_probs(a_mle, sigma_mle)
exp_counts = n * probs_hat

chi_stat = float(np.sum((freq - exp_counts) ** 2 / exp_counts))
df = 10 - 1 - 2
p_chi = float(st.chi2.sf(chi_stat, df))

print('Нормальная модель N(a, sigma^2), критерий хи-квадрат')
print(f'a = {a_mle:.6f}; sigma = {sigma_mle:.6f}')
print('Ожидаемые частоты:', np.round(exp_counts, 6))
print(f'chi^2 = {chi_stat:.6f}; df = {df}; p-value = {p_chi:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_chi > eps else 'отвергается')



# критерий Колмогорова
sample = np.repeat(np.arange(10), freq.astype(int))
sample = np.sort(sample)


def ks_stat_normal_with_refit(sample_arr: np.ndarray) -> float:

    m = float(sample_arr.mean())
    s = float(sample_arr.std(ddof=1))

    if s <= 0:
        return 0.0

    Fn_left = np.arange(len(sample_arr)) / len(sample_arr)
    Fn_right = np.arange(1, len(sample_arr) + 1) / len(sample_arr)
    F0 = st.norm.cdf(sample_arr, loc=m, scale=s)

    d = np.max(np.maximum(np.abs(F0 - Fn_left), np.abs(F0 - Fn_right)))
    return math.sqrt(len(sample_arr)) * d


delta_obs = ks_stat_normal_with_refit(sample)
m0 = float(sample.mean())
s0 = float(sample.std(ddof=1))

print()
print('Нормальная модель N(a, sigma^2), критерий Колмогорова')
print(f'a = {m0:.6f}; sigma = {s0:.6f}')
print(f'Δ = {delta_obs:.6f}')


# Параметрический бутстрап для Колмогорова
B = 50000
boot_stats = np.empty(B)

for b in range(B):
    y = np.sort(st.norm.rvs(loc=m0, scale=s0, size=n))
    boot_stats[b] = ks_stat_normal_with_refit(y)

p_ks = float(np.mean(boot_stats >= delta_obs))

print(f'p-value = {p_ks:.4f}')
print('Решение по H0:', 'нет оснований для отклонения гипотезы' if p_ks > eps else 'отвергается')

Нормальная модель N(a, sigma^2), критерий хи-квадрат
a = 5.289676; sigma = 2.679520
Ожидаемые частоты: [ 5.469818  5.507955  8.663355 11.87373  14.180665 14.757615 13.382777
 10.575134  7.2817    8.307251]
chi^2 = 9.802556; df = 7; p-value = 0.2000
Решение по H0: нет оснований для отклонения гипотезы

Нормальная модель N(a, sigma^2), критерий Колмогорова
a = 4.770000; sigma = 2.518036
Δ = 1.002094
p-value = 0.0156
Решение по H0: отвергается
